## Setup

The following packages are required to run the analysis. If not already installed, the packages will be installed using `pip`.

In [ ]:
!pip install numpy
!pip install pandas
!pip install scipy
!pip install hmmlearn
!pip install statsmodels
!pip install mpl_scatter_density
!pip install tqdm
!pip install colorama
!pip install pyliftover

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from tqdm import tqdm
import json, ast
import statsmodels.api as sm
import scipy.stats as stats
import math
import glob
from pyliftover import LiftOver
import requests
import warnings

# Suppress all warnings to keep the output clean
warnings.filterwarnings("ignore")

## Specify project directories

Define the paths to the data and results directories used in the project.

In [2]:
# Specify the directories containing data and results
data_path = '/oak/stanford/groups/mrivas/projects/wgs-constraint-llm/data/'
results_path = '/oak/stanford/groups/mrivas/projects/wgs-constraint-llm/osthoag/wgs-constraint-llm/results/'

# Specify the paths to the specific data files
# https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_44/gencode.v44.chr_patch_hapl_scaff.basic.annotation.gtf.gz
gene_annotation_file_path = data_path + 'gencode.v44.basic.annotation.gtf.gz'
scz_variants_file_path = data_path + 'scz.tsv.gz'
# https://genome.ucsc.edu/cgi-bin/hgTables?db=hg19&hgta_group=compGeno&hgta_track=allHg19RS_BW&hgta_table=allHg19RS_BW&hgta_doSchema=describe+table+schema
gerp_file_path = data_path + 'All_hg38_RS.bw'
# https://doi.org/10.6084/m9.figshare.27184245.v1
hmm_predictions_path = results_path + f'HMM_rgc_0.9_over20_chr2_predictions_rgc_wes.tsv.gz'
# https://zenodo.org/records/10813168/files/AlphaMissense_hg38.tsv.gz?download=1
alpha_missense_file_path = data_path + 'AlphaMissense_hg38.tsv.gz'

## Define helper methods

## Load Data
This section loads the gene annotation data from a GTF file and extracts relevant information such as gene ID, gene type, gene name, and transcript details. The data is then filtered to include only protein-coding regions.

In [ ]:
# Read the GTF file into a pandas DataFrame
gene_df = pd.read_csv(gene_annotation_file_path, sep='\t', comment='#', header=None, 
                      names=['chr', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute'], 
                      dtype={'start': int, 'end': int})

# Extract 'gene_id' from attributes
gene_df['gene_id'] = gene_df['attribute'].str.extract(r'gene_id "(.*?)"')

# Extract 'gene_type' from attributes
gene_df['gene_type'] = gene_df['attribute'].str.extract(r'gene_type "(.*?)"')

# Extract 'gene_name' from attributes
gene_df['gene_name'] = gene_df['attribute'].str.extract(r'gene_name "(.*?)"')

# Extract 'transcript_id' from attributes
gene_df['transcript_id'] = gene_df['attribute'].str.extract(r'transcript_id "(.*?)"')

# Extract 'transcript' and 'num' from transcript_id
gene_df[['transcript', 'transcript_num']] = gene_df['transcript_id'].str.split('.', expand=True)

# Extract 'transcript_name' from attributes
gene_df['transcript_name'] = gene_df['attribute'].str.extract(r'transcript_name "(.*?)"')

# Drop the original attribute column
gene_df = gene_df.drop('attribute', axis=1)

# Filter rows for protein-coding regions
gene_df = gene_df[(gene_df['gene_type'] == 'protein_coding') & (gene_df['feature'] == 'CDS')]

# Standardize the gene_id by removing any version numbers (i.e., text after the dot)
gene_df['std_gene_id'] = gene_df['gene_id'].str.split('.').str[0]

# Display the DataFrame
gene_df

## Load and Filter Schizophrenia Variant Data
This section reads in the schizophrenia variant data, splits the chromosome and position information, filters out unwanted chromosomes and variant types, and merges it with the gene annotation data to include gene names.

In [ ]:
# https://schema.broadinstitute.org/downloads
in_path  = data_path + "SCHEMA_variant_results.tsv.bgz"

# --- Load whole file ---
scz_variant_results_df = pd.read_csv(in_path, sep="\t", compression="gzip", low_memory=False, dtype={"locus":"string", "alleles":"string"})

# --- Split locus -> chr,pos ---
scz_variant_results_df[["chr","pos"]] = scz_variant_results_df["locus"].str.split(":", n=1, expand=True)
# normalize chr names to 'chrN'
need_chr = ~scz_variant_results_df["chr"].str.startswith("chr", na=False)
scz_variant_results_df.loc[need_chr, "chr"] = "chr" + scz_variant_results_df.loc[need_chr, "chr"].astype(str)
scz_variant_results_df["pos"] = pd.to_numeric(scz_variant_results_df["pos"], errors="coerce").astype("Int64")

# --- Parse ref/alt from JSON-like alleles column ---
def parse_alleles(s):
    if pd.isna(s): return pd.NA, pd.NA
    try:
        arr = json.loads(s)
    except Exception:
        arr = ast.literal_eval(s)
    if isinstance(arr, (list, tuple)) and len(arr) >= 2:
        return str(arr[0]), str(arr[1])
    return pd.NA, pd.NA

scz_variant_results_df[["ref","alt"]] = scz_variant_results_df["alleles"].apply(parse_alleles).apply(pd.Series)

# --- Liftover hg19 -> hg38 ---
lo = LiftOver("hg19", "hg38")

chrs = scz_variant_results_df["chr"].astype(str).to_numpy()
poses = scz_variant_results_df["pos"].to_numpy()

new_chr = np.empty(len(scz_variant_results_df), dtype=object)
new_pos = np.empty(len(scz_variant_results_df), dtype="object")  # keep object while we mark NAs
ok_mask = np.zeros(len(scz_variant_results_df), dtype=bool)

for i in tqdm(range(len(scz_variant_results_df)), desc="Lifting variants"):
    c = chrs[i]
    p = poses[i]
    if pd.isna(p):
        continue
    hit = lo.convert_coordinate(c, int(p))
    if hit:
        new_chr[i] = hit[0][0]
        new_pos[i] = int(hit[0][1])  # stays 0-based
        ok_mask[i] = True

# Keep only successfully lifted rows
scz_variant_results_df = scz_variant_results_df.loc[ok_mask].copy()
scz_variant_results_df["chr"] = new_chr[ok_mask]
scz_variant_results_df["pos"] = pd.Series(new_pos[ok_mask], index=scz_variant_results_df.index).astype("Int64")

# --- Put requested columns first and write ---
leading = ["chr", "pos", "ref", "alt"]
rest = [c for c in scz_variant_results_df.columns if c not in leading]  # keeps everything else
scz_variant_results_df[leading + rest].to_csv(data_path + "SCHEMA_variant_results_hg38.tsv.gz", sep="\t", index=False, compression="gzip")

print(f"Kept {len(scz_variant_results_df):,} lifted rows.")
scz_variant_results_df

## Load Constraint Predictions
Load saved HMM predictions for constraint data, which will be used in building the unified model.

In [ ]:
# GERP RS annotation.
#
# This cell previously inlined a bigWig lookup that indexed a 0-based value
# vector with 1-based HMM positions, shifting every score one base downstream.
# The lookup now lives in src/wgs_constraint/gerp.py, so the three notebooks
# that need it cannot drift apart again; the coordinate convention is an
# explicit argument there rather than an assumption.
#
# Loading the predictions is kept here because later cells use `pred`.
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd

from wgs_constraint import annotate_with_gerp

# --- load HMM predictions (once) ---
pred = pd.read_csv(hmm_predictions_path, sep="\t", dtype={"chr": "string"})
if "position" in pred.columns and "pos" not in pred.columns:
    pred = pred.rename(columns={"position": "pos"})
pred["pos"] = pred["pos"].astype(np.int64)
pred["chr"] = pred["chr"].astype("string")
pred_cols = pred.columns.tolist()

merged_constraint_df = annotate_with_gerp(
    pred,
    gerp_file_path,
    position_base=1,      # HMM positions are 1-based
)
merged_constraint_df["chr"] = merged_constraint_df["chr"].astype("category")
merged_constraint_df


## Load Missense Pathogenicity Data
Load and preprocess missense pathogenicity data, including extracting transcript details and renaming columns for consistency.

In [ ]:
# Read the missense pathogenicity data into a pandas DataFrame
alpha_missense_df = pd.read_csv(alpha_missense_file_path, sep='\t', header=3)

# Rename columns to standard labels
alpha_missense_df.rename(columns={"#CHROM": "chr", "POS": "pos", "REF": "ref", "ALT": 'alt'}, inplace=True)

# Extract 'transcript' and 'transcript_num' from transcript_id
alpha_missense_df[['transcript', 'transcript_num']] = alpha_missense_df['transcript_id'].str.split('.', expand=True)

# Display the DataFrame
alpha_missense_df

## Filter Variants for Analysis
Further filtering of variants is done to include only those with a total allele count (ac_ctrl + ac_case) of 5 or less, and with non-zero allele numbers in both cases and controls.

In [ ]:
variants_df = pd.read_csv(data_path + "SCHEMA_variant_results_hg38.tsv.gz", sep="\t", compression="gzip")

filter_consequence_list = [
#     "3_prime_UTR_variant",
#     "5_prime_UTR_variant",
    "coding_sequence_variant",
#     "downstream_gene_variant",
#     "intergenic_variant",
#     "intron_variant",
    "mature_miRNA_variant",
#      "non_coding_transcript_exon_variant",
#     "non_coding_transcript_variant",
    "null",
    "synonymous_variant",
#    "upstream_gene_variant",
#     "splice_region_variant",
]

pLoF_consequence_list = [
    'stop_gained',
    'splice_acceptor_variant',
    'splice_donor_variant',
    'frameshift_variant'
]

missense_consequence_list = [
    'inframe_insertion',
    'inframe_deletion',
    'stop_retained_variant',
    'stop_lost',
    'missense_variant_mpc_<2',
    'protein_altering_variant',
    'missense_variant_mpc_2-3',
    'missense_variant_mpc_>=3'
]

# Filter out rows related to sex chromosomes and unrelated consequences
variants_df = variants_df[(variants_df['chr'] != "chrX") &
                          (variants_df['chr'] != "chrY") &
                          (variants_df['chr'] != "chrMT")]
variants_df = variants_df[~variants_df['consequence'].isin(filter_consequence_list)]
variants_df = variants_df[~variants_df['consequence'].isna()]

# Merge the filtered variants with gene names from the gene annotation data
variants_df = pd.merge(variants_df, gene_df[['std_gene_id', 'gene_name']].drop_duplicates(), left_on='gene_id', right_on='std_gene_id', how='left').drop('std_gene_id', axis=1)

# Filter variants where the sum of allele counts in cases and controls is less than or equal to 5
# and where both case and control allele numbers are non-zero
variants_df = variants_df[(variants_df['ac_ctrl'] + variants_df['ac_case'] <= 5) &
                          (variants_df['an_case'] > 0) &
                          (variants_df['an_ctrl'] > 0)]

# Calculate effect size and variance for variants
ALT_AJ = variants_df['ac_case']
ALT_ExAC = variants_df['ac_ctrl']
REF_AJ = variants_df['an_case'] - variants_df['ac_case']
REF_ExAC = variants_df['an_ctrl'] - variants_df['ac_ctrl']

# Logarithmic transformation of effect size calculation
variants_df['effect_size'] = np.log(((0.5 + ALT_AJ) * (0.5 + REF_ExAC)) / ((0.5 + REF_AJ) * (0.5 + ALT_ExAC)))

# Variance calculation for the effect size
variants_df['var_effect_size'] = (1 / (0.5 + REF_AJ) + 1 / (0.5 + REF_ExAC) + 1 / (0.5 + ALT_AJ) + 1 / (0.5 + ALT_ExAC))

# Add indicator variable for pLoF variants
variants_df['pLoF_ind'] = (variants_df['consequence'].isin(pLoF_consequence_list)).astype('int32')

# Add indicator variable for missense variants
variants_df['missense_ind'] = (variants_df['consequence'].isin(missense_consequence_list)).astype('int32')

variants_df

## Build Unified Constraint, Pathogenicity, and pLoF Model
This section creates a unified model that integrates constraint predictions, pathogenicity predictions, and loss-of-function (pLoF) indicators to assess the association with schizophrenia.

In [ ]:
# Subset the constraint predictions to include only relevant columns
constraint_subset_df = merged_constraint_df[['chr', 'pos', 'prob_0', 'GERP_RS']]

# Subset the AlphaMissense predictions to include only relevant columns
am_subset_df = alpha_missense_df[['chr', 'pos', 'ref', 'alt', 'am_pathogenicity']]

# Subset the pLoF dataframe to include only useful columns
variants_subset_df = variants_df[['chr', 'pos', 'ref', 'alt', 'gene_id', 'gene_name', 'group', 'ac_case', 'an_case', 'ac_ctrl', 'an_ctrl', 'pLoF_ind', 'missense_ind', 'effect_size', 'var_effect_size']]

# Merge constraint predictions, pathogenicity predictions, and pLoF variants based on chromosome and position
constraint_pathogenicity_pLoF_df = pd.merge(constraint_subset_df, pd.merge(variants_subset_df, am_subset_df, on=['chr', 'pos', 'ref', 'alt'], how='left'), on=['chr', 'pos'], how='inner')

# Save the merged dataframe to a compressed CSV file for further analysis
constraint_pathogenicity_pLoF_df.to_csv(results_path + f"constraint_gerp_am_scz_variants.tsv.gz", index=False, compression='gzip', sep='\t')

# Display the DataFrame
constraint_pathogenicity_pLoF_df

In [ ]:
# Read the data from the file
input_df = pd.read_csv(results_path + f'constraint_gerp_am_scz_variants.tsv.gz', sep='\t')

# frequency threshold
# input_df = input_df[input_df['AF'] <= 0.05]

# Ensure probabilities are not exactly 1 or 0
epsilon = 1e-2
input_df['prob_0'] = np.clip(input_df['prob_0'], epsilon, 1 - epsilon)
input_df['am_pathogenicity'] = np.clip(input_df['am_pathogenicity'], epsilon, 1 - epsilon)

# Impute missing values with column means
input_df[['prob_0', 'GERP_RS', 'am_pathogenicity']] = input_df[['prob_0', 'GERP_RS', 'am_pathogenicity']].fillna(input_df[['prob_0', 'GERP_RS', 'am_pathogenicity']].mean())
input_df['pLoF_ind'] = input_df['pLoF_ind'].fillna(0)
input_df['missense_ind'] = input_df['missense_ind'].fillna(0)

# Apply log transformations
# input_df[['log_constraint', 'log_gerp', 'log_pathogenicity']] = -np.log1p(-(input_df[['prob_0', 'GERP_RS', 'am_pathogenicity']]))
input_df[['log_constraint', 'log_pathogenicity']] = -np.log1p(-(input_df[['prob_0', 'am_pathogenicity']]))

# Remove rows with missing effect_size or var_effect_size
input_df = input_df.dropna(subset=['effect_size', 'var_effect_size'])
input_df = input_df[input_df['var_effect_size'] != 0]

# Display table of inputs to meta-regression model
input_df

In [ ]:
from statsmodels.regression.linear_model import WLS
from statsmodels.tools.tools import add_constant
    
# Group data by gene
grouped_gene_data = input_df.groupby(['gene_id', 'gene_name', 'group'])

# Initialize lists to store results
meta_model_results = []

# Loop over each gene group and build a meta-regression model
for gene_key, gene_data in tqdm(grouped_gene_data, desc="Processing genes", unit="gene"):
    gene_id, gene_name, group = gene_key

    if gene_data[['log_constraint', 'GERP_RS', 'log_pathogenicity', 'pLoF_ind', 'missense_ind', 'effect_size', 'var_effect_size']].isnull().any().any():
        continue

    # Meta-regression model for the gene
    X = add_constant(gene_data[['log_constraint', 'GERP_RS', 'log_pathogenicity', 'pLoF_ind', 'missense_ind']])
    y = gene_data['effect_size']
    weights = 1 / gene_data['var_effect_size']

    try:
        model = WLS(y, X, weights=weights, missing='drop').fit()

        # Append relevant results to the meta_model_results list
        meta_model_results.append({
            'gene_id': gene_id,
            'gene_name': gene_name,
            'group': group,
            'n_variants': len(gene_data),
            'p_constraint': model.pvalues['log_constraint'],
            'p_gerp': model.pvalues['GERP_RS'],
            'p_pathogenicity': model.pvalues['log_pathogenicity'],
            'p_pLoF': model.pvalues['pLoF_ind'],
            'p_missense': model.pvalues['missense_ind'],
    #         'p_const': model.pvalues['const'],
            'p_unified': model.f_pvalue
        })
        
    except Exception as e:
#         print(f"Error processing {gene_key}: {str(e)}")
        pass

# Create a DataFrame from the results
unified_model_df = pd.DataFrame(meta_model_results)

# Save the results to a compressed CSV file
unified_model_df.to_csv(results_path + "schizophrenia_variants_unified_gerp_model_pvalues.tsv", index=False, sep='\t')

# Display the contents of the DataFrame
unified_model_df

In [ ]:
# Read the saved model results
unified_model_df = pd.read_csv(results_path + "schizophrenia_variants_unified_gerp_model_pvalues.tsv", sep='\t')

# Filter for n_variants
min_variants = 25
unified_model_df = unified_model_df[unified_model_df['n_variants'] >= min_variants]

# Load the corresponding publication results for the current group
results_pub_df = pd.read_csv(data_path + "SCHEMA_gene_results.tsv.bgz", sep='\t', compression="gzip")

# Merge the meta-regression results with the publication data on gene_id
merged_pub_df = pd.merge(
    unified_model_df,
    results_pub_df,
    on=['gene_id', 'group'],
)

# Compute -log10 of p-values for plotting
merged_pub_df['minus_log_pval'] = -np.log10(merged_pub_df['p_unified'])
merged_pub_df['minus_log_pub_pval'] = -np.log10(merged_pub_df['P meta'])

# Replace Inf values with NaN for plotting
merged_pub_df.replace([np.inf, -np.inf], np.nan, inplace=True)

for group, group_pub_df in merged_pub_df.groupby('group'):
    # Drop rows with NaN or infinite values in either axis
    clean_df = group_pub_df[
        group_pub_df[['minus_log_pub_pval', 'minus_log_pval']].applymap(np.isfinite).all(axis=1)
    ]

    # Determine plotting bounds
    min_pval_x = clean_df['minus_log_pub_pval'].min()
    min_pval_y = clean_df['minus_log_pval'].min()
    max_pval_x = clean_df['minus_log_pub_pval'].max()
    max_pval_y = clean_df['minus_log_pval'].max()

    # Safe fallback
    if np.isnan(min_pval_x): min_pval_x = 0
    if np.isnan(min_pval_y): min_pval_y = 0
    if np.isnan(max_pval_x): max_pval_x = 1
    if np.isnan(max_pval_y): max_pval_y = 1

    # Define bin edges
    num_bins = 10
    x_edges = np.linspace(min_pval_x, max_pval_x + 1, num_bins + 1)
    y_edges = np.linspace(min_pval_y, max_pval_y + 1, num_bins + 1)

    # Set up the plot
    plt.figure(figsize=(8, 6))

    # Plot 2D histogram
    plt.hist2d(
        clean_df['minus_log_pub_pval'],
        clean_df['minus_log_pval'],
        bins=[x_edges, y_edges],
        cmap='Blues',
        norm=LogNorm()
    )

    # Add diagonal line
    plt.plot([0, max(max_pval_x, max_pval_y) + 1], [0, max(max_pval_x, max_pval_y) + 1], color='red', linestyle='--')

    # Select genes to annotate
    annot_df = clean_df[
        (abs(clean_df['minus_log_pub_pval'] - clean_df['minus_log_pval']) > 2) &
        (clean_df[['minus_log_pub_pval', 'minus_log_pval']].max(axis=1) > 4)
    ].copy()

    # Digitize values to assign them to bins
    annot_df['x_bin_idx'] = np.digitize(annot_df['minus_log_pub_pval'], bins=x_edges) - 1
    annot_df['y_bin_idx'] = np.digitize(annot_df['minus_log_pval'], bins=y_edges) - 1

    # Calculate bin centers for annotation
    x_bin_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_bin_centers = 0.5 * (y_edges[:-1] + y_edges[1:])

    # Compute y-axis range for dynamic stacking offset
    y_range = max_pval_y + 1 - min_pval_y
    stack_offset = y_range * 0.02  # 2% of y-range

    # Group by bin indices and stack vertically (centered)
    for (x_idx, y_idx), group_df in annot_df.groupby(['x_bin_idx', 'y_bin_idx']):
        if 0 <= x_idx < len(x_bin_centers) and 0 <= y_idx < len(y_bin_centers):
            x_center = x_bin_centers[x_idx]
            y_center = y_bin_centers[y_idx]
            total_height = (len(group_df) - 1) * stack_offset
            group_df_sorted = group_df.sort_values(by='minus_log_pval')
            for i, (_, row) in enumerate(group_df_sorted.iterrows()):
                plt.text(
                    x_center,
                    y_center - total_height / 2 + i * stack_offset,
                    row['gene_name'],
                    fontsize=8,
                    ha='center'
                )

    # Set axis limits and labels
    plt.xlim(min_pval_x, max_pval_x + 1)
    plt.ylim(min_pval_y, max_pval_y + 1)

    plt.xlabel(r'SCHEMA $-log_{10}(p)$', fontsize=12)
    plt.ylabel(r'Unified Model $-log_{10}(p)$', fontsize=12)
    plt.title(f'Unified Model vs SCHEMA p-values for schizophrenia', fontsize=14, fontweight="bold")

    plt.colorbar(label='Log-Scaled Count')

    # Save or show the plot
    plt.savefig(results_path + f"Figure A1: p-value comparison for schizophrenia")
    plt.show()

In [4]:
# Combine SCHEMA data with our predictions for full comparison of results
comparison_df = merged_pub_df[['gene_id', 'gene_name', 'group', 'n_variants', 'p_constraint', 'p_gerp', 'p_pathogenicity', 'p_pLoF', 'p_missense', 'p_unified', 'P meta']]
column_names = {
    'gene_id': 'Gene ID',
    'gene_name': 'Gene Name',
    'group': 'Group',
    'n_variants': '# of Variants',
    'p_constraint': 'Constraint p-value',
    'p_gerp': 'GERP p-value',
    'p_pathogenicity': 'Pathogenicity p-value',
    'p_pLoF': 'pLoF p-value',
    'p_missense': 'Missense p-value',
    'p_unified': 'Unified Model p-value',
    'P meta': 'SCHEMA p-value'
}
comparison_df.rename(columns=column_names, inplace=True)
comparison_df.to_csv(results_path + f"table_A2_schizophrenia_full_p_value_comparison_n{min_variants}.csv", index=False)

In [ ]:
# Read saved results
comparison_df = pd.read_csv(results_path + f"table_A2_schizophrenia_full_p_value_comparison_n{min_variants}.csv")

# Filter the DataFrame for significant p-values or specific gene names for closer examination
suggestive_mask = (comparison_df['Unified Model p-value'] < 1e-4)

# Display the filtered DataFrame sorted by the unified p-value
pd.set_option('display.max_rows', 100)
(
    comparison_df[suggestive_mask]
    .drop(columns=['Gene ID'])  # drop the Gene ID column
    .sort_values('Unified Model p-value')
    .style
    .format({col: "{:.2e}" for col in comparison_df.columns if 'p-value' in col})
    .hide(axis='index')  # hide the index column
)